# Spatial Metabolomics: QC, Annotation & Differential Analysis

This notebook documents the full processing pipeline for spatial metabolomics data (MALDI-MSI) collected from tissue sections at **Day 1** and **Day 60** post-treatment. Positive and negative ion panels are processed in parallel throughout.

**Pipeline overview:**
1. Helper functions
2. Load & process negative ion panel
3. Load & process positive ion panel
4. METASPACE spectral annotation (FDR < 0.05)
5. Normalization, batch correction & dimensionality reduction
6. Cluster-level differential metabolite analysis (Leiden)
7. Transfer H&E-based cell type annotations
8. Cell-type-level differential metabolite analysis (Wilcoxon)
9. Day60 vs Day1 differential analysis per cell type (Wilcoxon + volcano plots)
10. Pseudobulk DESeq2 — Day60 vs Day1 per cell type

**Cell type labels** (from H&E image segmentation):  
`epithelial`, `adipocyte`, `blood_vessel`, `stromal` (stromal/other cells, previously labeled `unknown`), `nohne` (outside H&E mask — excluded from all biological analyses).

> **File naming note:** Raw input files on disk have swapped filenames relative to their biological content (positive/negative labels are inverted). This is corrected in code below with explicit comments wherever it applies.

## Imports

In [ ]:
from glob import glob
import re
import warnings

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors
import seaborn as sns

import scanpy as sc
import anndata as ad
import torch
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

from metaspace import SMInstance

warnings.simplefilter(action='ignore', category=FutureWarning)
mpl.rcParams['pdf.fonttype'] = 42
sc._settings.settings._vector_friendly = True

## Configuration

Set all file paths here before running the notebook.

In [ ]:
# ── Input data paths ──────────────────────────────────────────────────────────
# NOTE: raw CSV filenames on disk are mislabeled (pos↔neg swapped at save time).
# Variable names throughout this notebook use the correct biological labels.
RAW_NEG_CSV    = '../data/merged_db_pos.csv'       # disk name 'pos' → negative panel data
RAW_NEG_NA_CSV = '../data/Nna_meta_pos.csv'
RAW_POS_CSV    = '../data/merged_db_neg.csv'       # disk name 'neg' → positive panel data
RAW_POS_NA_CSV = '../data/Nna_meta_neg.csv'

METASPACE_SAMPLE_SHEET = '../data/metadb_spmeta.csv'

# Intermediate / checkpoint files (written and re-read to avoid re-running METASPACE queries)
ADATA_NEG_ANNOTATED = '../data/adata_neg_metaspace_annotated.h5ad'
ADATA_POS_ANNOTATED = '../data/adata_pos_metaspace_annotated.h5ad'
ADATA_NEG_FINAL     = '../data/adata_neg_final.h5ad'
ADATA_POS_FINAL     = '../data/adata_pos_final.h5ad'

# Previously computed H&E cell type annotations (from companion H&E analysis notebook)
ADATA_NEG_HNE = '../data/adata_neg_hne_annotated.h5ad'
ADATA_POS_HNE = '../data/adata_pos_hne_annotated.h5ad'

# ── Output paths ───────────────────────────────────────────────────────────────
RESULTS_DIR = '../results/'

# ── Analysis parameters ───────────────────────────────────────────────────────
FDR_CUTOFF       = 0.05   # METASPACE annotation FDR threshold
NA_FILTER_MAX    = 12     # max missing pixels allowed per metabolite
CELLTYPES        = ['epithelial', 'adipocyte', 'blood_vessel', 'stromal']

---
## 1. Helper Functions

In [ ]:
def get_neutral_molecule(mol, element, change=1):
    """
    Add or remove one atom of `element` from a molecular formula string.

    Parameters
    ----------
    mol : str
        Molecular formula, e.g. 'C10H11N4O5'.
    element : str
        Atom symbol to modify, e.g. 'H', 'Na', 'K', 'Cl'.
    change : int
        +1 to add, -1 to remove.

    Returns
    -------
    str
        Modified molecular formula.
    """
    elem_pattern = r'(' + re.escape(element) + r')(\d*)'
    elem_match = re.search(elem_pattern, mol)
    if elem_match:
        count = int(elem_match.group(2)) if elem_match.group(2) else 1
        new_count = count + change
        if new_count > 0:
            new_formula = re.sub(elem_pattern, f'{element}{new_count}', mol)
        else:
            new_formula = re.sub(elem_pattern, '', mol)
        if new_count == 1:
            new_formula = new_formula.replace(f'{element}1', element)
    return new_formula


def parse_adducts_to_neutral(df_na, mode):
    """
    Parse adduct ion strings into (neutral formula, ion, n_na) table.

    For negative mode: [M-H]-, [M-Cl]- → remove adduct atom.
    For positive mode: [M+H]+, [M+Na]+, [M+K]+ → remove adduct atom.
    Cl adducts in both modes have been manually verified in the source table.

    Parameters
    ----------
    df_na : pd.DataFrame
        Table with columns ['metabolites', 'n_na'].
    mode : str
        'pos' or 'neg'.

    Returns
    -------
    pd.DataFrame
        Columns: ['mol_formula', 'ion', 'neutral_mol', 'n_na'].
    """
    df = pd.DataFrame(
        [a.split('-') if '-' in a else a.split('+') for a in df_na.metabolites],
        columns=['mol_formula', 'ion']
    )
    df['n_na'] = df_na.n_na.values

    for i, r in df.iterrows():
        df.loc[i, 'neutral_mol'] = get_neutral_molecule(r.mol_formula, r.ion, change=-1)

    return df


def filter_zero_var_per_batch(adata, batch_key):
    """
    Remove metabolites with zero variance in any batch.

    Zero-variance features cause divide-by-zero in ComBat. Applied before batch correction.

    Parameters
    ----------
    adata : AnnData
    batch_key : str
        Key in adata.obs identifying batches.

    Returns
    -------
    AnnData
        Filtered copy with zero-variance-per-batch features removed.
    """
    X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
    keep = np.ones(adata.n_vars, dtype=bool)
    for b in adata.obs[batch_key].unique():
        mask = (adata.obs[batch_key] == b).values
        keep &= (np.var(X[mask], axis=0) > 0)
    n_removed = (~keep).sum()
    print(f"  Removed {n_removed} zero-variance-per-batch features, keeping {keep.sum()}.")
    return adata[:, keep].copy()

---
## 2. Load & Process Negative Ion Panel

> **File naming note:** The raw file `merged_db_pos.csv` on disk contains negative ion panel data (filename was saved with swapped label). Variable names use the correct biological labels.

In [ ]:
# Load raw intensity matrix and missingness table for the negative ion panel
cts_neg_raw = pd.read_csv(RAW_NEG_CSV)
df_na_neg   = pd.read_csv(RAW_NEG_NA_CSV)

In [ ]:
# Parse adduct ions to neutral molecular formulas (negative mode: remove adduct atom)
df_adducts_neg = parse_adducts_to_neutral(df_na_neg, mode='neg')
print("Unique ions in negative panel:", df_adducts_neg.ion.unique())
df_adducts_neg.head()

In [ ]:
# Assign neutral formulas as row index
cts_neg_raw.index = df_adducts_neg.neutral_mol.values

# Filter: keep metabolites missing in ≤ NA_FILTER_MAX pixels
cts_neg_filt = cts_neg_raw.loc[df_adducts_neg[df_adducts_neg.n_na <= NA_FILTER_MAX].neutral_mol].fillna(0)
print(f"After NA filter: {cts_neg_filt.shape[0]} metabolites × {cts_neg_filt.shape[1]} pixels")

# Collapse duplicate neutral formulas (multiple adducts → same molecule) by summing intensities
cts_neg = cts_neg_filt.groupby(cts_neg_filt.index).sum()
print(f"After deduplication: {cts_neg.shape[0]} metabolites × {cts_neg.shape[1]} pixels")

In [ ]:
# Build AnnData (obs = pixels, var = metabolites)
adata_neg = ad.AnnData(cts_neg.T)

# Extract spatial coordinates and sample metadata from pixel names
# Format: x{X}_y{Y}_pid{PID}_day{DAY}_{panel}
adata_neg.obs['pid']      = [a.split('_')[2] for a in adata_neg.obs_names]
adata_neg.obs['batch']    = [a.split('_')[2] + '_' + a.split('_')[3] for a in adata_neg.obs_names]
adata_neg.obs['type']     = [a.split('_')[3] for a in adata_neg.obs_names]
adata_neg.obs['x_pixel']  = [int(a.split('x')[1].split('_')[0]) for a in adata_neg.obs_names]
adata_neg.obs['y_pixel']  = [int(a.split('y')[1].split('_')[0]) for a in adata_neg.obs_names]
adata_neg.obs['panel']    = 'neg'

# Preserve raw counts before any normalization
adata_neg.layers['counts'] = adata_neg.X.copy()

print(adata_neg)
print(f"Max raw intensity: {adata_neg.X.max():.1f}")

---
## 3. Load & Process Positive Ion Panel

> **File naming note:** The raw file `merged_db_neg.csv` on disk contains positive ion panel data (filename was saved with swapped label).

In [ ]:
# Load raw intensity matrix and missingness table for the positive ion panel
cts_pos_raw = pd.read_csv(RAW_POS_CSV)
df_na_pos   = pd.read_csv(RAW_POS_NA_CSV)

In [ ]:
# Parse adduct ions to neutral molecular formulas
# Positive mode adducts: [M+Na]+, [M+H]+, [M+K]+ (Cl cases manually verified)
df_adducts_pos = parse_adducts_to_neutral(df_na_pos, mode='pos')
print("Unique ions in positive panel:", df_adducts_pos.ion.unique())
df_adducts_pos.head()

In [ ]:
# Assign neutral formulas as row index
cts_pos_raw.index = df_adducts_pos.neutral_mol.values

# Filter: keep metabolites missing in ≤ NA_FILTER_MAX pixels
cts_pos_filt = cts_pos_raw.loc[df_adducts_pos[df_adducts_pos.n_na <= NA_FILTER_MAX].neutral_mol].fillna(0)
print(f"After NA filter: {cts_pos_filt.shape[0]} metabolites × {cts_pos_filt.shape[1]} pixels")

# Collapse duplicate neutral formulas by summing
cts_pos = cts_pos_filt.groupby(cts_pos_filt.index).sum()
print(f"After deduplication: {cts_pos.shape[0]} metabolites × {cts_pos.shape[1]} pixels")

In [ ]:
# Build AnnData
adata_pos = ad.AnnData(cts_pos.T)

adata_pos.obs['pid']      = [a.split('_')[2] for a in adata_pos.obs_names]
adata_pos.obs['batch']    = [a.split('_')[2] + '_' + a.split('_')[3] for a in adata_pos.obs_names]
adata_pos.obs['type']     = [a.split('_')[3] for a in adata_pos.obs_names]
adata_pos.obs['x_pixel']  = [int(a.split('x')[1].split('_')[0]) for a in adata_pos.obs_names]
adata_pos.obs['y_pixel']  = [int(a.split('y')[1].split('_')[0]) for a in adata_pos.obs_names]
adata_pos.obs['panel']    = 'pos'

adata_pos.layers['counts'] = adata_pos.X.copy()

print(adata_pos)
print(f"Max raw intensity: {adata_pos.X.max():.1f}")

---
## 4. METASPACE Spectral Annotation

For each sample, we query METASPACE for annotations passing FDR < 0.05. Each annotation is converted from its adduct form to a neutral molecular formula using `get_neutral_molecule`.

The per-sample annotations are merged by **union**: a metabolite is considered annotated if it passes FDR < 0.05 in at least one sample.

Two summary metrics are added to `adata.var` for each panel:
- `n_samples_annotated` — number of samples (out of 8) where it passed FDR < 0.05
- `mean_msm` — mean MSM score across annotated samples
- `metaspace_annotated` — boolean flag (`n_samples_annotated >= 1`)

> These are metadata only — no features are removed here. Use `metaspace_annotated` or `n_samples_annotated` to filter downstream as needed.

> **Panel column mapping:** The METASPACE sample sheet uses the same swapped filename convention as the raw CSVs (`pos` column → negative ion panel datasets, `neg` column → positive ion panel datasets). The `panel_col` arguments in `fetch_metaspace_annotations` are set accordingly. Update if your sample sheet uses corrected column names.

In [ ]:
sm = SMInstance()

In [ ]:
def fetch_metaspace_annotations(metadb, panel_col, fdr=FDR_CUTOFF):
    """
    Fetch METASPACE annotations for all samples in one panel and
    return a summary DataFrame indexed by neutral molecular formula.

    Parameters
    ----------
    metadb : pd.DataFrame
        Sample sheet with columns ['slide_id', 'pid', 'time', 'pos', 'neg'].
        Each row is one slide; 'pos' and 'neg' are METASPACE dataset IDs.
    panel_col : str
        Which column to use: 'pos' or 'neg' (note swapped convention, see above).
    fdr : float
        FDR threshold for annotation filtering.

    Returns
    -------
    pd.DataFrame
        Index = neutral_mol formula.
        Columns: n_samples_annotated (int), mean_msm (float), metaspace_annotated (bool).
    """
    records = []

    for _, row in metadb.iterrows():
        sid   = row['slide_id']
        ds_id = row[panel_col]
        ds    = sm.dataset(id=ds_id)
        print(f"  Fetching {panel_col} | slide {sid} | dataset {ds_id}")

        results = ds.results().reset_index()
        sig     = results[results['fdr'] <= fdr][['formula', 'adduct', 'msm']].copy()

        for _, r in sig.iterrows():
            records.append({
                'neutral_mol': r['formula'],
                'msm':         r['msm'],
                'sample_id':   sid,
            })

    df = pd.DataFrame(records)

    summary = (
        df.groupby('neutral_mol')
          .agg(
              n_samples_annotated=('sample_id', 'nunique'),
              mean_msm=('msm', 'mean'),
          )
          .reset_index()
          .set_index('neutral_mol')
    )
    summary['metaspace_annotated'] = True

    print(f"  → {len(summary)} unique neutral formulas annotated (union across samples)")
    return summary

In [ ]:
# Load sample sheet and fetch annotations for both panels
metadb = pd.read_csv(METASPACE_SAMPLE_SHEET)
print(metadb.head())
print(f"\n{len(metadb)} samples total")

In [ ]:
# Fetch annotations
# panel_col='pos' → negative panel datasets (swapped convention, see note above)
# panel_col='neg' → positive panel datasets
print("=== Negative panel ===")
annot_neg = fetch_metaspace_annotations(metadb, panel_col='pos')

print("\n=== Positive panel ===")
annot_pos = fetch_metaspace_annotations(metadb, panel_col='neg')

In [ ]:
# Summary: number of formulas annotated across samples
for label, annot in [('neg', annot_neg), ('pos', annot_pos)]:
    print(f"Panel {label}: {len(annot)} annotated neutral formulas")
    print(annot['n_samples_annotated'].value_counts().sort_index().rename('n_formulas').to_frame())
    print()

### Join annotation metadata onto `adata.var`

Features not found in METASPACE get `n_samples_annotated = 0` and `metaspace_annotated = False`.

In [ ]:
def annotate_adata_var(adata, annot_df):
    """
    Left-join METASPACE annotation summary onto adata.var.
    Missing metabolites get n_samples_annotated=0, mean_msm=NaN, metaspace_annotated=False.
    """
    adata.var = pd.DataFrame(index=adata.var_names)
    adata.var = adata.var.join(
        annot_df[['n_samples_annotated', 'mean_msm', 'metaspace_annotated']],
        how='left'
    )
    adata.var['n_samples_annotated'] = adata.var['n_samples_annotated'].fillna(0).astype(int)
    adata.var['metaspace_annotated'] = adata.var['metaspace_annotated'].fillna(False)
    n_annot = adata.var['metaspace_annotated'].sum()
    print(f"  {n_annot}/{adata.n_vars} features have METASPACE annotation (FDR < {FDR_CUTOFF})")
    return adata

adata_neg = annotate_adata_var(adata_neg, annot_neg)
adata_pos = annotate_adata_var(adata_pos, annot_pos)

In [ ]:
# Visualize annotation coverage per panel
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (label, adata) in zip(axes, [('Negative panel', adata_neg), ('Positive panel', adata_pos)]):
    counts = adata.var['n_samples_annotated'].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_xlabel('# samples annotated (out of 8)')
    ax.set_ylabel('# metabolites')
    ax.set_title(label)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'metaspace_annotation_coverage.pdf', dpi=150)
plt.show()

In [ ]:
# Save annotated AnnData objects (checkpoint before filtering)
adata_neg.write(ADATA_NEG_ANNOTATED)
adata_pos.write(ADATA_POS_ANNOTATED)

In [ ]:
# Reload from checkpoint (skip METASPACE query on re-runs)
adata_neg = sc.read_h5ad(ADATA_NEG_ANNOTATED)
adata_pos = sc.read_h5ad(ADATA_POS_ANNOTATED)

In [ ]:
# Restrict to METASPACE-annotated features only
adata_neg = adata_neg[:, adata_neg.var.metaspace_annotated].copy()
adata_pos = adata_pos[:, adata_pos.var.metaspace_annotated].copy()

print("Neg panel after annotation filter:", adata_neg)
print("Pos panel after annotation filter:", adata_pos)

---
## 5. Normalization, Batch Correction & Dimensionality Reduction

Applied identically to each panel:
1. `normalize_total` (library-size normalization)
2. `log1p` transformation
3. ComBat batch correction (key = `batch` = patient × timepoint)
4. PCA → neighbors → UMAP → Leiden clustering

PCA is run before and after ComBat as a QC check for batch effect removal.

### 5a. Negative Panel

In [ ]:
# Restore raw counts to .X before normalization
adata_neg.X = adata_neg.layers['counts'].copy()

# Normalize and log-transform
sc.pp.normalize_total(adata_neg)
sc.pp.log1p(adata_neg)

# PCA before batch correction (QC check)
sc.tl.pca(adata_neg)
sc.pl.pca(adata_neg, color=['batch', 'batch'], dimensions=[(0,1),(2,3)],
          ncols=2, size=2, wspace=.5, title=['PC1v2 pre-ComBat', 'PC3v4 pre-ComBat'])

In [ ]:
# ComBat batch correction across patient-day batches
sc.pp.combat(adata_neg, key='batch')

# PCA after batch correction
sc.tl.pca(adata_neg)
sc.pl.pca(adata_neg, color=['batch', 'batch'], dimensions=[(0,1),(2,3)],
          ncols=2, size=2, wspace=.5, title=['PC1v2 post-ComBat', 'PC3v4 post-ComBat'])

### 5b. Positive Panel

In [ ]:
# Restore raw counts to .X before normalization
adata_pos.X = adata_pos.layers['counts'].copy()

sc.pp.normalize_total(adata_pos)
sc.pp.log1p(adata_pos)

sc.tl.pca(adata_pos)
sc.pl.pca(adata_pos, color=['batch', 'batch'], dimensions=[(0,1),(2,3)],
          ncols=2, size=2, wspace=.5, title=['PC1v2 pre-ComBat', 'PC3v4 pre-ComBat'])

In [ ]:
sc.pp.combat(adata_pos, key='batch')

sc.tl.pca(adata_pos)
sc.pl.pca(adata_pos, color=['batch', 'batch'], dimensions=[(0,1),(2,3)],
          ncols=2, size=2, wspace=.5, title=['PC1v2 post-ComBat', 'PC3v4 post-ComBat'])

In [ ]:
# Save normalized + batch-corrected objects (before H&E annotation)
adata_neg.write(ADATA_NEG_ANNOTATED.replace('.h5ad', '_norm_combat.h5ad'))
adata_pos.write(ADATA_POS_ANNOTATED.replace('.h5ad', '_norm_combat.h5ad'))

---
## 6. Cluster-level Differential Metabolite Analysis

Wilcoxon rank-sum test: each Leiden cluster vs. all others. Visualized as dot plots showing top 10 upregulated metabolites per cluster. Two Leiden resolutions are compared (0.3 and default) to assess robustness of cluster structure.

### 6a. Negative Panel — Leiden clusters

In [ ]:
for res_key in ['leiden_0p3', 'leiden']:
    sc.tl.rank_genes_groups(adata_neg, groupby=res_key, method='wilcoxon')
    sc.tl.dendrogram(adata_neg, groupby=res_key)
    sc.pl.rank_genes_groups_dotplot(adata_neg, values_to_plot='logfoldchanges',
                                    cmap='bwr', vcenter=0,
                                    title=f'Neg panel — {res_key}')

### 6b. Positive Panel — Leiden clusters

In [ ]:
for res_key in ['leiden_0p3', 'leiden']:
    sc.tl.rank_genes_groups(adata_pos, groupby=res_key, method='wilcoxon')
    sc.tl.dendrogram(adata_pos, groupby=res_key)
    sc.pl.rank_genes_groups_dotplot(adata_pos, values_to_plot='logfoldchanges',
                                    cmap='bwr', vcenter=0,
                                    title=f'Pos panel — {res_key}')

---
## 7. Transfer H&E-based Cell Type Annotations

Cell type labels (`celltype_byhne`) were derived from H&E image segmentation in a companion analysis and are transferred here by pixel index alignment.

**Cell type categories:**
| Label | Description |
|-------|-------------|
| `epithelial` | Epithelial cells |
| `adipocyte` | Adipocytes |
| `blood_vessel` | Vascular/endothelial regions |
| `stromal` | Stromal and other non-classified cells (previously labeled `unknown` in H&E annotation files) |
| `nohne` | Pixels outside the H&E mask — excluded from all biological analyses |

In [ ]:
# Load previously annotated objects (H&E-based cell type labels)
adata_pos_prev = sc.read(ADATA_POS_HNE)
adata_neg_prev = sc.read(ADATA_NEG_HNE)

In [ ]:
# Align obs indices between new and previous objects (panel suffix correction)
adata_neg.obs.index = [a.replace('pos', 'neg') for a in adata_neg.obs.index]
adata_pos.obs.index = [a.replace('neg', 'pos') for a in adata_pos.obs.index]

mask_n = list(set(adata_neg_prev.obs.index) & set(adata_neg.obs.index))
mask_p = list(set(adata_pos_prev.obs.index) & set(adata_pos.obs.index))
print(f"Matching pixels — neg: {len(mask_n)}, pos: {len(mask_p)}")

In [ ]:
# Transfer annotation by matching obs index
adata_neg.obs.loc[mask_n, 'celltype_byhne'] = adata_neg_prev.obs.loc[mask_n, 'celltype_byhne']
adata_pos.obs.loc[mask_p, 'celltype_byhne'] = adata_pos_prev.obs.loc[mask_p, 'celltype_byhne']

# Rename 'unknown' → 'stromal' (stromal/other cells; see table above)
for adata in [adata_neg, adata_pos]:
    adata.obs['celltype_byhne'] = adata.obs['celltype_byhne'].replace({'unknown': 'stromal'})

print("Neg panel cell types:", adata_neg.obs.celltype_byhne.unique().tolist())
print("Pos panel cell types:", adata_pos.obs.celltype_byhne.unique().tolist())

del adata_pos_prev, adata_neg_prev

In [ ]:
# Subset to pixels with H&E coverage (exclude 'nohne')
adata_neg = adata_neg[adata_neg.obs.celltype_byhne.notna() & (adata_neg.obs.celltype_byhne != 'nohne')].copy()
adata_pos = adata_pos[adata_pos.obs.celltype_byhne.notna() & (adata_pos.obs.celltype_byhne != 'nohne')].copy()

print(f"Neg after removing nohne pixels: {adata_neg.n_obs:,} pixels")
print(f"Pos after removing nohne pixels: {adata_pos.n_obs:,} pixels")

In [ ]:
# Save final annotated objects
adata_neg.write(ADATA_NEG_FINAL)
adata_pos.write(ADATA_POS_FINAL)

---
## 8. Cell-type-level Differential Metabolite Analysis

Wilcoxon rank-sum test: each cell type vs. all others. Top 10 upregulated metabolites per cell type are visualized as dot plots and all results saved to CSV.

In [ ]:
def run_celltype_de(adata, panel_label):
    """Wilcoxon DE per cell type vs. rest; return filtered significant results."""
    adata.uns['log1p']['base'] = None
    sc.tl.rank_genes_groups(adata, groupby='celltype_byhne',
                             group=CELLTYPES, method='wilcoxon')
    df = sc.get.rank_genes_groups_df(adata, group=CELLTYPES, pval_cutoff=0.05)

    # Top 10 upregulated metabolites per cell type
    dex_d = {}
    for ct in df.group.unique():
        top = (df.query("group == @ct and logfoldchanges > 0 and pvals_adj <= 0.05")
                 .sort_values('scores', ascending=False)
                 .head(10)['names'].tolist())
        dex_d[ct] = top

    sc.tl.dendrogram(adata, groupby='celltype_byhne')
    dp = sc.pl.dotplot(adata, dex_d, groupby='celltype_byhne', dendrogram=True,
                       return_fig=True, standard_scale='var', cmap='Reds')
    dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds')
    dp.savefig(RESULTS_DIR + f'dotplot_top10DE_bycelltype_{panel_label}.pdf', dpi=300)

    return df

In [ ]:
df_neg_celltype = run_celltype_de(adata_neg, 'neg')
df_neg_celltype.to_csv(RESULTS_DIR + 'DE_spmetabo_foreachcelltype_neg.csv', index=False)
df_neg_celltype.head()

In [ ]:
df_pos_celltype = run_celltype_de(adata_pos, 'pos')
df_pos_celltype.to_csv(RESULTS_DIR + 'DE_spmetabo_foreachcelltype_pos.csv', index=False)
df_pos_celltype.head()

---
## 9. Day60 vs Day1 Differential Analysis per Cell Type (Wilcoxon)

Within each cell type, Day60 vs Day1 is compared using a Wilcoxon rank-sum test.  
Results are saved to CSV and visualized as volcano plots.

> **Statistical note:** This test treats each pixel independently and does not account for the paired patient structure. Use the pseudobulk DESeq2 results (Section 10) for statistically rigorous inference that properly models patient-level variation.

In [ ]:
def run_day_de_wilcoxon(adata, panel_label):
    """
    Within each cell type, Wilcoxon test Day60 vs Day1.
    Returns combined DataFrame with celltype column.
    """
    adata.uns['log1p']['base'] = None
    df_all = []
    for ct in CELLTYPES:
        adata_sub = adata[adata.obs.celltype_byhne == ct].copy()
        sc.tl.rank_genes_groups(adata_sub, groupby='type',
                                reference='day1', method='wilcoxon')
        df = sc.get.rank_genes_groups_df(adata_sub, group='day60', pval_cutoff=None)
        df['celltype'] = ct
        df_all.append(df)

    df_all = pd.concat(df_all, ignore_index=True)

    # Dot plot: top 10 upregulated at Day60 per cell type
    dex_d = {}
    for ct in df_all.celltype.unique():
        top = (df_all.query("celltype == @ct and logfoldchanges > 0 and pvals_adj <= 0.05")
                     .sort_values('scores', ascending=False)
                     .head(10)['names'].tolist())
        dex_d[ct] = top

    dp = sc.pl.dotplot(adata, dex_d, groupby='type', dendrogram=True,
                       return_fig=True, standard_scale='var', cmap='Reds')
    dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds')
    dp.savefig(RESULTS_DIR + f'dotplot_top10DE_day60vsday1_percelltype_{panel_label}.pdf', dpi=300)

    return df_all

In [ ]:
df_neg_day = run_day_de_wilcoxon(adata_neg, 'neg')
df_neg_day.to_csv(RESULTS_DIR + 'DE_day60vsday1_percelltype_neg.csv', index=False)
df_neg_day.head()

In [ ]:
df_pos_day = run_day_de_wilcoxon(adata_pos, 'pos')
df_pos_day.to_csv(RESULTS_DIR + 'DE_day60vsday1_percelltype_pos.csv', index=False)
df_pos_day.head()

### Volcano plots — Day60 vs Day1 per cell type

In [ ]:
def plot_de_volcano(df_DE, cap_fdr, cap_lfc, charge, verbose=True):
    """
    Volcano plot of Day60 vs Day1 DE results, one panel per cell type.

    Parameters
    ----------
    df_DE : pd.DataFrame
        Output of run_day_de_wilcoxon.
    cap_fdr : float
        -log10(FDR) axis cap for display.
    cap_lfc : float
        logFC axis cap for display. Points beyond cap are plotted at the boundary.
    charge : str
        'pos' or 'neg' — used for output filename.
    """
    if verbose:
        print(f"Volcano plot capped at LFC ±{cap_lfc}, -log10(FDR) {cap_fdr}")

    df_DE = df_DE.copy()
    df_DE['sig'] = 'ns'
    df_DE.loc[(df_DE.pvals_adj <= 0.05) & (df_DE.logfoldchanges > 0),  'sig'] = 'lightup'
    df_DE.loc[(df_DE.pvals_adj <= 0.05) & (df_DE.logfoldchanges < 0),  'sig'] = 'lightdown'
    df_DE.loc[(df_DE.pvals_adj <= 0.05) & (df_DE.logfoldchanges >= 0.5), 'sig'] = 'up'
    df_DE.loc[(df_DE.pvals_adj <= 0.05) & (df_DE.logfoldchanges <= -0.5), 'sig'] = 'down'

    df_DE['logFDR'] = -np.log10(df_DE['pvals_adj'])
    df_DE['logFDR_adj'] = df_DE['logFDR'].clip(upper=cap_fdr)
    df_DE.loc[(df_DE['logFDR'] > cap_fdr) & (df_DE.logfoldchanges >= 0.5),  'sig'] = 'up_capped'
    df_DE.loc[(df_DE['logFDR'] > cap_fdr) & (df_DE.logfoldchanges <= -0.5), 'sig'] = 'down_capped'

    df_DE['lfc_adj'] = df_DE['logfoldchanges'].clip(lower=-cap_lfc, upper=cap_lfc)
    df_DE.loc[(df_DE['logfoldchanges'] > cap_lfc) & (df_DE.pvals_adj <= 0.05), 'sig'] = 'up_capped'
    df_DE.loc[(df_DE['logfoldchanges'] < -cap_lfc) & (df_DE.pvals_adj <= 0.05), 'sig'] = 'down_capped'

    cols_sig = {
        'up': 'red', 'up_capped': 'darkred',
        'down': 'blue', 'down_capped': 'darkblue',
        'lightup': '#F5594E', 'lightdown': '#5240F5',
        'ns': 'grey'
    }

    fig, axes = plt.subplots(2, 2, figsize=(6, 6))
    plt.subplots_adjust(wspace=.5, hspace=.5)
    for ax, ct in zip(axes.flat, CELLTYPES):
        df = df_DE.query('celltype == @ct')
        sns.scatterplot(data=df, x='lfc_adj', y='logFDR_adj',
                        hue='sig', palette=cols_sig, legend=False,
                        rasterized=True, linewidth=0, ax=ax)
        ax.axvline(0, color='grey', linestyle='--')
        ax.axhline(-np.log10(0.05), color='grey', linestyle='--')
        ax.set_xlabel('logFC')
        ax.set_ylabel('-log10(FDR)')
        ax.set_title(ct)

    plt.savefig(RESULTS_DIR + f'volcano_day60vsday1_bycelltype_{charge}.pdf', dpi=300)
    plt.show()
    return df_DE

In [ ]:
_ = plot_de_volcano(df_pos_day, cap_fdr=50, cap_lfc=10, charge='pos')
_ = plot_de_volcano(df_neg_day, cap_fdr=50, cap_lfc=10, charge='neg')

---
## 10. Pseudobulk DESeq2 — Day60 vs Day1 per Cell Type

Pixels are aggregated (summed) per `batch × celltype` to create pseudobulk samples, which are then used as input to DESeq2. This approach properly accounts for biological replication at the patient level.

DESeq2 is run per cell type with design formula `~type` (Day1 as reference).

> **Note:** Patient ID (`pid`) is not included in the current design formula. Consider `~pid + type` if paired structure should be explicitly modeled.

In [ ]:
def run_pseudobulk_deseq2(adata, panel_label):
    """
    Pseudobulk (sum per batch × celltype) + DESeq2 Day60 vs Day1.
    Returns combined results DataFrame with metabolite names resolved.
    """
    # Use raw integer counts for DESeq2
    adata_raw = adata.copy()
    adata_raw.X = adata_raw.layers['counts'].astype(int)

    pdata = dc.pp.pseudobulk(
        adata=adata_raw,
        sample_col='batch',
        groups_col='celltype_byhne',
        mode='sum',
    )

    # QC: sample composition
    dc.pl.obsbar(adata=pdata, y='celltype_byhne', hue='pid', figsize=(6, 3))

    # Normalize for PCA visualization only (counts restored after)
    pdata.layers['counts'] = pdata.X.copy()
    sc.pp.normalize_total(pdata, target_sum=1e4)
    sc.pp.log1p(pdata)
    sc.pp.scale(pdata, max_value=10)
    sc.tl.pca(pdata)
    dc.pp.swap_layer(adata=pdata, key='counts', inplace=True)

    sc.pl.pca_variance_ratio(pdata)
    sc.pl.pca(pdata, color=['celltype_byhne', 'type', 'batch'], ncols=1, size=300)

    # DESeq2 per cell type
    df_all = []
    for ct in CELLTYPES:
        pdata_sub = pdata[pdata.obs.celltype_byhne == ct]
        inference = DefaultInference(n_cpus=8)
        dds2 = DeseqDataSet(
            counts=pdata_sub.X,
            metadata=pdata_sub.obs[['type', 'pid']],
            design='~type',
            refit_cooks=True,
            inference=inference,
        )
        # Use integer indices during fitting; map back to names after
        dds2.var.index = range(dds2.n_vars)
        dds2.fit_size_factors()
        dds2.fit_genewise_dispersions()
        dds2.fit_dispersion_trend()
        dds2.fit_dispersion_prior()
        dds2.fit_MAP_dispersions()
        dds2.fit_LFC()
        dds2.refit_cooks = False

        stat_res = DeseqStats(dds2, contrast=['type', 'day60', 'day1'],
                              cooks_filter=False, independent_filter=True)
        stat_res.summary()

        df = stat_res.results_df.copy()
        df['celltype_byhne'] = ct
        df_all.append(df)

    df_all = pd.concat(df_all)

    # Map integer indices back to metabolite names
    id_to_name = {i: pdata.var_names[i] for i in range(pdata.n_vars)}
    df_all['names'] = df_all.index.map(id_to_name)

    return df_all

In [ ]:
df_neg_pb = run_pseudobulk_deseq2(adata_neg, 'neg')
df_neg_pb.to_csv(RESULTS_DIR + 'DE_day60vsday1_percelltype_pseudobulk_neg.csv')
df_neg_pb.head()

In [ ]:
df_pos_pb = run_pseudobulk_deseq2(adata_pos, 'pos')
df_pos_pb.to_csv(RESULTS_DIR + 'DE_day60vsday1_percelltype_pseudobulk_pos.csv')
df_pos_pb.head()